# Hourly Delay Profile (Polars)

Shows the local hours when buses are most late using the Polars analysis path.

In [1]:
from pathlib import Path
import importlib.util
import sys

import plotly.express as px
import polars as pl

PROJECT_ROOT = Path.cwd().resolve()
for candidate in (PROJECT_ROOT, *PROJECT_ROOT.parents):
    if (candidate / "pyproject.toml").exists() and (candidate / "analysis" / "polars").exists():
        PROJECT_ROOT = candidate
        break
else:
    raise RuntimeError("Run this notebook from the project root or a subdirectory.")

POLARS_ANALYSIS_DIR = PROJECT_ROOT / "analysis" / "polars"
polars_analysis_path = str(POLARS_ANALYSIS_DIR)
if polars_analysis_path in sys.path:
    sys.path.remove(polars_analysis_path)
sys.path.insert(0, polars_analysis_path)

for module_name in ("_shared", "report_cache", "cli_common"):
    module = sys.modules.get(module_name)
    module_file = getattr(module, "__file__", None) if module else None
    module_path = Path(module_file).resolve() if module_file else None
    if module_path is not None and POLARS_ANALYSIS_DIR not in module_path.parents:
        sys.modules.pop(module_name, None)

def load_polars_script(module_name: str, filename: str):
    spec = importlib.util.spec_from_file_location(module_name, POLARS_ANALYSIS_DIR / filename)
    module = importlib.util.module_from_spec(spec)
    assert spec.loader is not None
    spec.loader.exec_module(module)
    return module

hourly = load_polars_script("polars_hourly_delay_profile", "hourly-delay-profile.py")

DB = PROJECT_ROOT / "data" / "foli.db"
CACHE_DIR = PROJECT_ROOT / "outputs" / "polars-report-cache"
FORCE_CACHE = False
NO_CACHE = False
TIMEZONE = "Europe/Helsinki"
LINE_REF = None
MIN_OBSERVATIONS = 30
LIMIT = 24
QUALITY_MODE = "conservative"
BUCKET = "trip-stop"


Set `NO_CACHE = True` to read SQLite directly, or `FORCE_CACHE = True` to rebuild the Polars cache.

In [2]:
class Args:
    db = DB
    cache_dir = CACHE_DIR
    force_cache = FORCE_CACHE
    no_cache = NO_CACHE
    timezone = TIMEZONE
    line_ref = LINE_REF
    min_observations = MIN_OBSERVATIONS
    limit = LIMIT
    quality_mode = QUALITY_MODE
    exclude_stop_call_disagreement = False
    bucket = BUCKET

buckets = hourly.load_delay_buckets_for_args(Args)
profile = hourly.build_hourly_delay_profile(
    buckets,
    line_ref=LINE_REF,
    min_observations=MIN_OBSERVATIONS,
    limit=LIMIT,
)
profile


hour_local,bucket_count,raw_poll_count,signed_mean_delay_min,median_delay_min,p75_delay_min,p90_delay_min,p95_delay_min,pct_over_3_min_late,pct_over_5_min_late,pct_early,pct_over_1_min_early,pct_over_3_min_early
str,u32,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""15:00""",141140,377516,0.95,0.5,2.23,4.58,6.42,18.34,8.6,36.22,18.66,5.66
"""16:00""",133864,348649,0.91,0.42,2.12,4.48,6.28,17.69,8.1,37.61,19.29,5.29
"""14:00""",132401,359401,0.6,0.26,1.74,3.77,5.57,14.02,6.01,40.35,21.04,6.14
"""13:00""",116522,312177,0.56,0.28,1.7,3.62,5.25,13.22,5.47,39.45,20.72,6.53
"""12:00""",103334,275904,0.45,0.18,1.48,3.23,4.78,11.23,4.59,41.18,21.05,5.9
…,…,…,…,…,…,…,…,…,…,…,…,…
"""06:00""",95593,250197,-0.42,-0.15,0.65,1.75,2.63,3.7,0.91,54.15,29.58,8.91
"""00:00""",43241,105137,-0.92,-0.63,0.26,1.47,2.56,4.01,1.84,65.77,41.8,15.48
"""04:00""",6380,18607,-0.14,0.0,0.62,1.22,1.68,1.0,0.33,44.17,19.58,3.57


Delay has positive and negative values. Positive values mean the bus is late; negative values mean the bus is early.

In [3]:
if profile.is_empty():
    print("No matching observations found.")
else:
    fig = px.bar(
        profile.sort("hour_local"),
        x="hour_local",
        y="p90_delay_min",
        title="P90 delay by local hour",
        labels={"hour_local": "Local hour", "p90_delay_min": "P90 delay, minutes"},
    )
    fig.update_layout(showlegend=False)
    fig.show()
